# Lecture 28 - Rate of Change Along a River: `np.gradient`
### EES 3350/5350 · Python in Earth Science · Wednesday, November 11, 2026 

On Monday, `linregress` gave us **one** rate for a whole stretch of data, with its uncertainty. Today, we find the rate at **every point**. Remember calculus? This is the **derivative**. With data, it's one NumPy function: `np.gradient`.

Our "axis" today isn't time, it's **distance**. Alonga  river, the rate of change of elevation with distance is the river's **slope**: how many meters it drops for every meter it flows. Slope controls how fast the water moves and how much sediment the river can carry. 

**Learning Objectives**
- explain a derivative of data as "change in $y$ / change in $x$" between neighboring points
- use `np.gradient(y, x)` to get the rate of change at every point, and say how it differs from `.diff()`
- get the units right (km vs m)
- see why derivatives make bumpy data look **bumpier**, and smooth them with `.rolling()`

---
## Part 0 : set up

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---
## Part 1 : a river's longitudinal profile

A river's **longitudinal profile** is its elevation from upstream to the mount. Our data: the lower **Cedar River**, Washington, surveyed by the USGS in 2010 form Landsburg down to Renton, where it flows into Lake Washington. There's a point every **10 meters** for 36 kilometers.

<span style="font-size:0.85em;color:#6b7a8d;"><em>Data: Konrad, C.P., 2017, Longitudinal profile and off-channel habitats of the lower Cedar River, Washington: U.S. Geological Survey data release, https://doi.org/10.5066/F7K935R2.</em></span>

In [ ]:
!pwd

In [ ]:
!head -3 ../../course-materials/week12/LowerCedarLongProfile.csv

There are lots of columns. We're only going to grab 3 of them:
|**column**|**what it means**|
|--|--|
|`riverkm`|distance **upstream** from the mouth, in kilometers|
|`bed_elev.m`| elevation of the **river bed**, in meters|
|`wselev.m`| elevation of the **water surface** in meters|

The column names have dots in them, which makes them awkward to type.

<div class="alert alert-info">

### Exercise 1 : processing data

1. Read in the CSV.
2. Grab only the three columns in the table above.
3. Rename them.
4. Drop a few rows where an elevation is missing (`NA`).
    
</div>

In [ ]:
# 1 - import the csv
cedar = _______________ 

# 2 - make a mask to grab only the three columns listed above
cedar = _______________

# 3 - rename the columns
cedar = _______________

# 4 - drop NA values
cedar = _______________

print(cedar.shape)
cedar.head()

Once you have the data processed, let's plot it to see what's going on:

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(cedar["riverkm"], cedar["bed_m"], color="sienna", linewidth=0.8, label="river bed")
ax.plot(cedar["riverkm"], cedar["water_m"], color="#0072B2", linewidth=1.5, label="water surface")

ax.invert_xaxis()    # NEW
ax.set_xlabel("distance upstream from the mouth (km)")
ax.set_ylabel("elevation (m)")
ax.set_title("Lower Cedar River, Washington (2010)")
ax.legend();

##### New piece: `ax.invert_xaxis()` 
>This flips the x-axis so the river **flows from left to right**, from Landsburg (about 36 km, about 157 m) to Renton (near 0 m). At this scale, the bed and water surface look like one line.

We can zoom into a single kilometer to see the difference:

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(cedar["riverkm"], cedar["bed_m"], color="sienna", linewidth=0.8, label="river bed")
ax.plot(cedar["riverkm"], cedar["water_m"], color="#0072B2", linewidth=1.5, label="water surface")

ax.set_xlim(21, 20)
ax.set_ylim(70.5, 76)
ax.set_xlabel("distance upstream from the mouth (km)")
ax.set_ylabel("elevation (m)")
ax.set_title("Lower Cedar River, Washington (2010)")
ax.legend();

##### Note: setting the limits with the higher number *first* keeps the river flowing left to right.

---
## Part 2 : slope = rise / run, at every point

A rate of change is always **change in $y$ / change in $x$**: rise over run, just like the slope of a line.

First, the units. `riverkm` is in kilometers, but elevation is in meters, so  make a distance column in **meters**

In [ ]:
cedar["distance_m"] = cedar["riverkm"] * 1000

### By hand, for one point

What's the slope **at** the second point (position [1])? The fair answer: look on step to **each side** of it, and divide the change in elevation by the distance between them. That's called a **centered difference**:
$$ \text{slope at point 1} = \frac{z_2 - z_0}{x_2 - x_0}$$

Let's pull the columns out as arrays so we can index them:

In [ ]:
z = np.array(cedar["water_m"])
x = np.array(cedar["distance_m"])

by_hand = (z[2] - z[0]) / (x[2] - x[0])
print(round(by_hand,3))

### `np.gradient` does that at every point

`np.gradient(y, x)` gives pack **one slope per point**, the same length as the data.

In [ ]:
cedar["water_slope"] = np.gradient(cedar["water_m"], cedar["distance_m"])

cedar[["riverkm", "water_m", "water_slope"]].head()

Look at the second row's `water_slope`: it matches our hand calculation.

At the very first and last points there's no neighbor on one side, so it uses the one neighbor it has.

The result is in **meters of elevation per meter of river**. A slope of 0.002 means the river drops 2 m for every 1000 m it flows.
> The numbers are mostly positive because `riverkm` increases **upstream**, the direction the river is going up. A few are slightly negative: the elevations are only given to the nearest centimeter, and over 20 m, a 1-cm rounding error looks like a small slope.)

### How is this different from `.diff()`?
Earlier, you saw `.diff()`, which gives the change **between** neighbors:

In [ ]:
cedar["water_m"].diff().head()

|**tool**|**what it gives**|**how many values**|
|--|--|--|
|`.diff()`| change in $y$ **between** neighbors (no dividing)| one fewer than the data (the first is `NaN`)|
|`np.gradient(y, x)`| change in $y$ / by change in $x$, **at** each point| same as the data|

`np.gradient` divides by the spacing between your `x` values. That's what turns "meters" into "meters **per meter**". It also means your points don't have to be evenly spaced: when `dropna()` removed a few rows, `np.gradient` still divided by the real distance between the neighbors that were left.

<div class="alert alert-danger">

**Common Error:** Units! If you write `np.gradient(cedar["water_m"], cedar["riverkm"])`, you get **meters per kilometer**, 1,000 times bigger than meters per meter. Neither is wrong, but you have to know which one you have. **Units of a derivative = units of $y$ / units of $x$.**

</div>

<div class="alert alert-info">

### Exercsie 2 : the bed's slope

Make a column `bed_slope` the same way, then run the cell to compare the two slopes.
    
</div>

In [ ]:
# your answer here
cedar["bed_slope"] = ________________________________

fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
axes[0].plot(cedar["riverkm"], cedar["water_slope"], color="#0072B2", linewidth=0.8)
axes[0].set_ylabel("water-surface slope")
axes[1].plot(cedar["riverkm"], cedar["___"], color="sienna", linewidth=0.8)
axes[1].set_ylabel("bed slope")

for ax in axes:
    ax.axhline(0, color="black", linewidth=0.8)
axes[1].invert_xaxis()
axes[1].set_xlabel("distance upstream from the mouth (km)");

<div class="alert alert-warning">

**Question:** The bed slope is often **negative**. That would mean the bed goes **up** as you go downstream. How can that be, in a river that flows downhill? And why is the water surface so much smoother?

</div>

> your answer here

<details>
<summary style="color:green;"><strong>Answer</strong></summary>

**Pools.** The bed dips into a deep pool and then climbs back up over a riffle on the downstream side. Over that short climb, the bed really does slope "uphill" downstream. The water still flows downhill, because the **water surface** keeps dropping. Water fills the pools and smooths over the bumps, so its slope is positive almost everywhere (about 94% of points, against only about 65% for the bed).

The bed profile looked smooth on the full-river plot, but its derivative is wild. **Derivatives make small bumps look big.** That's why geomorphologists usually measure a river's slope from the water surface, or over long distances.

</details>

<div class="alert alert-info">

### Exercise 3 : where is the river the steepest?

Smooth the water-surface slope over **1 km** with a centered moving average and covert it to **meters per kilometer**. The points are 10 m apart, so 1 km is 100 points.
    
</div>

In [ ]:
smooth = cedar["water_slope"].rolling(___, center=True).mean()
cedar["slope_m_per_km"] = smooth * ____

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(cedar["riverkm"], cedar["slope_m_per_km"], color="#0072B2", linewidth=2)

ax.invert_xaxis()
ax.set_xlabel("distance upstream from the mouth (km)")
ax.set_ylabel("slope (m drop per km)");

<details>
<summary style="color:green;"><strong>What the answer means.</strong></summary>

Upstream, near Landsburg, the river drops about **6-7 m per km**. In the last few kilometers before the mouth, it drops only about **1-2 m per km**. It's bumpy in between, but the overall trend is clear: the river gets **less steep downstream**. That's the **concave-up** profile from Lecture 22. As more water joins the river downstream, it can move its sediment on a gentler slope.

</details>

<div class="alert alert-success">

**Tip:** The USGS data file already has a bed-slope column, `bedgrad`. Its values match your `bed_slope` (to within rounding), because the USGS calculated it the same way: a centered difference over 10-m spacing. When you can, check your calculation against a published one.

</div>

---
## Part 3 : reach slopes with uncertainty

`np.gradient` gives a slope at every point. For a **reach** (a stretch of river), geomorphologists often want one slope with an uncertainty (think about `linregress`).

We can do that by:

1. **Splitting the river into 5-km reaches** (0-5, 5-10, .., 30-36.1 km) and fitting `water_m` vs. `distance_m` with `stats.linregress`

In [ ]:
from scipy import stats

edges = [0, 5, 10, 15, 20, 25, 30, 36.1]
middles = []
slopes = []
errors = []

for i in range(len(edges)-1):
    start = edges[i]
    end = edges[i + 1]
    in_reach = (cedar["riverkm"] >= start) & (cedar["riverkm"] < end)
    reach = cedar[in_reach]

    r = stats.linregress(reach["distance_m"], reach["water_m"])
    middles.append((start + end) /2)
    slopes.append(r.slope * 1000)
    errors.append(2 * r.stderr * 1000)


2. **Plotting each reach's slope** in m/km with 2 standard errors

In [ ]:
fig, ax = plt.subplots(figsize=(7, 2.5))
ax.errorbar(middles, slopes, yerr=errors, fmt="o", capsize=5, color="black")
ax.invert_xaxis()
ax.set_xlabel("distance upstream (km)")
ax.set_ylabel("reach slope (m/km)")

for m, s, e in zip(middles, slopes, errors):
    print(f"{m:5.1f} km: {s:.2f} ± {e:.3f} m/km")

The reach slopes climb from aout 2.1 m/km near the mouth to about 6.0 m/km upstream, with a dip int he 15-20 km reach (3.5 m/km). The `±` values are tiny, so almost none of the error bars overlap.

But those uncertainties are **too small to believe**. `linregress` thinks it has about 500 independent poitns per reach, but neighboring points 10 m apart aren't independent: they're on the same pool or riffle sometimes. The real uncertainty is bigger. 

---
## A useful table

|**you want**|**write**|
|:--|:--|
|rate of change at every point|`np.gradient(y, x)`|
|change between neighbors (no dividing)|`df["COL"].diff()`|
|one rate for a whole stretch, with uncertainty|`stats.linregress(x, y)`|
|smooth a noisy rate|`.rolling(N, center=True).mean()`|
|convert km to m|`df["km"] * 1000`|
|flip the x-axis (river flows left to right)|`ax.invert_xaxis()`|

**Units of a derivative = units of $y$ ÷ units of $x$.** m ÷ m = m/m. m ÷ km = m/km. Check them every time.
